In [1]:
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

# 1. 고해상도 설정 (맥 레티나 디스플레이 대응)
%config InlineBackend.figure_format = 'retina'

# 2. 맥 전용 나눔고딕 설정 
# 시스템에 나눔고딕이 설치되어 있다면 경로 지정 없이 이름만으로도 설정 가능합니다.
plt.rc('font', family='NanumBarunGothic') # 또는 'NanumBarunGothic'

# 3. 마이너스 기호(-) 깨짐 방지 (한글 폰트 적용 시 필수)
plt.rcParams['axes.unicode_minus'] = False


print("맥(Mac) 환경 나눔고딕 설정 완료! 슝=3")


맥(Mac) 환경 나눔고딕 설정 완료! 슝=3


In [2]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import matplotlib.pyplot as plt

import re
import os
import io
import time
import random
import math

import seaborn # Attention 시각화를 위해 필요!

print(torch.__version__)

2.3.0


In [ ]:

def positional_encoding(pos, d_model):
    def cal_angle(position, i):
        return position / np.power(10000, (i // 2) * 2 / d_model)

    def get_posi_angle_vec(position):
        return [cal_angle(position, i) for i in range(d_model)]

    sinusoid_table = np.array([get_posi_angle_vec(pos_i) for pos_i in range(pos)])
    sinusoid_table[:, 0::2] = np.sin(sinusoid_table[:, 0::2])
    sinusoid_table[:, 1::2] = np.cos(sinusoid_table[:, 1::2])
    return sinusoid_table

print("슝=3")

슝=3


In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super(MultiHeadAttention, self).__init__()
        self.num_heads = num_heads # 8
        self.d_model = d_model # 512

        # 각 머리(Head)가 담당할 차원 수를 계산 (예: 512차원 / 8개 머리 = 머리당 64차원)
        self.depth = d_model // self.num_heads # 소숫점 버리기

        # Q, K, V를 만들기 위한 선형 층 (가중치 행렬)
        self.W_q = nn.Linear(d_model, d_model)  # Linear Layer
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)

        # 여러 머리의 결과를 합친 후 마지막으로 정리해주는 선형 층
        self.linear = nn.Linear(d_model, d_model)

    def scaled_dot_product_attention(self, Q, K, V, mask):
        """실제 어텐션 점수를 계산하는 핵심 함수 (d_model=512, heads=8 기준)"""
        # [현재 입력 상태] Q, K, V는 이미 머리가 쪼개진 4차원 데이터 (batch_size, 8, 50, 64)
        # 배치(예:16문장) > 2. 머리(8개전문가) > 3. 문장(50개단어) > 4. 특징(64개피쳐)

        #  Q, K, V 는 가중치가 곱해진 '데이터(Activation)
        # [현재 시점 Q, K, V의 차원] : (batch_size, num_heads, seq_len, depth)
        # 예시: (16, 8, 50, 64) 👉 16개 문장, 8개 머리, 50개 단어, 머리당 64개 특징
        # [ # 1단계: 배치(16개 문장)
        #     [ # 2단계: 머리(8개의 서로 다른 시선)
        #         [ # 3단계: 문장(한 문장 속 50개 단어)
        #             [0.1, 0.2, ... 64개], # 4단계: 한 단어의 특징 일부(64개)
        #             ... (총 50개 단어 나열)
        #         ], 
        #         ... (총 8개 머리 존재)
        #     ],
        #     ... (총 16개 문장 존재)
        # ]

        # d_k: 한 머리가 담당하는 특징의 개수 (64)
        d_k = K.size(-1) 

        # 1. Q와 K를 점곱하여 유사도(Score) 계산 (QK^T)
        # K.transpose(-2, -1) 👉 마지막 두 차원을 뒤집어 (batch, heads, 64, 50)으로 변환
        # torch.matmul(Q, ...) 👉 50x64와 64x50을 곱해 (batch, heads, 50, 50)의 점수판 생성
        # 1. "피쳐들은 없어지고" 👉 [정보의 압축]
        # 단어 하나하나가 가진 *64개의 복잡한 특징(피쳐)**들은 서로 곱해지고 더해지면서 사라집니다.
        # 대신 그 피쳐들이 힘을 합쳐서 **'유사도'를 나타내는 '숫자 하나'로 변신한 것입니다.
        # 2. "Row는 Q, Col은 K" 👉 [주체와 대상의 확립]
        # 행렬 곱셈의 결과물인 (50, 50) 표에서:

        # 행(Row): Q의 단어들 (질문자 50개)
        # 열(Col): K의 단어들 (응답자 50개)
        # 👉 "내가(Q) 너를(K) 얼마나 중요하게 생각하나?"라는 구도가 명확해집니다.
        QK = torch.matmul(Q, K.transpose(-2, -1))

        # 2. 스케일링: 차원 수의 루트값(sqrt(64)=8)으로 나누어 점수가 너무 비대해지는 것 방지
        # (batch, heads, 50, 50)
        scaled_qk = QK / torch.sqrt(torch.tensor(d_k, dtype=torch.float32))

        # 3. 마스킹: 미래 정보를 예측하지 못하도록 -10억(절벽)으로 덮어버림
        # mask * -1e9: 가릴 곳(1)은 -10억, 보여줄 곳(0)은 0이 됨
        # scaled_qk + 마스크: 가릴 곳의 점수만 바닥(-10억)으로 떨어뜨림
        if mask is not None:
            scaled_qk = scaled_qk.masked_fill(mask == 1, -1e9)

        # 4. 소프트맥스: 정규화된 점수를 확률(0~1)로 변환 (합계 1)
        # -10억이었던 자리는 확률이 0%가 됨
        # 가장 마지막 차원(가로 방향)을 기준으로 소프트맥스를 계산
        # 0번 차원 (16): 배치
        # 1번 차원 (8): 머리
        # 2번 차원 (50): 행 Q(Row - 어떤 단어로부터)
        # 3번(-1) 차원 (50): 열 K(Column - 어떤 단어에게)
        # 우리가 원하는 건? "한 단어(행)가 다른 50개 단어(열)에게 어텐션을 어떻게 나눠주는지"입니다. 그래서 -1, 
        # 만약 2라면 세로방향, 1이면 머리방향  X
        # Attention Distributions
        attentions = F.softmax(scaled_qk, dim=-1)
        
        # 5. 최종 출력: 확률에 따라 실제 데이터(V)를 가중합산
        # (batch, heads, 50, 50) * (batch, heads, 50, 64)
        # (batch, heads, 50, 64)
        out = torch.matmul(attentions, V)


        return out, attentions

    def split_heads(self, x):
        """head 분할"""
        # (16, 50, 512), (batch_size, Sequence Length, Embedding Dimension )
        batch_size = x.size(0) # 16
        # view는 shape만 변환...
        # -1 알아서 채워넣어라...
        x = x.view(batch_size, -1, self.num_heads, self.depth) # (16, 50, 8, 64)
        # 차원의 순서를 맞바꾸는 자리바꿈"
        # 바로 GPU의 병렬 처리 방식 때문입니다.
        # (16, 50, 8, 64) 👉 (16, 8, 50, 64) (Batch Size, Num Heads, Sequence Length[토큰갯수], d_k)
        x = x.permute(0, 2, 1, 3)

        return x

    def combine_heads(self, x):
        """split_heads가 했던일을 역순으로 한다.."""
        batch_size = x.size(0)
        x = x.permute(0, 2, 1, 3) #(16, 8, 50, 64) ->  (16, 50, 8, 64)
        # x.contiguous는 permute로 차원 순서를 마구 뒤섞으면, 겉으로 보기엔 순서가 바뀌었지만 컴퓨터 메모리 상의 실제 데이터 위치는 엉켜있는 상태가 됩니다.
        # x.permute다음연산이 View라서 반드시 해야한다.
        x = x.contiguous().view(batch_size, -1, self.d_model)

        return x

    def forward(self, Q, K, V, mask=None):
        # Q, K, V (16, 50, 512)
        # 가중치 곱
        WQ = self.W_q(Q)
        WK = self.W_k(K)
        WV = self.W_v(V)

        #(16, 8, 50, 64) (Batch Size, Num Heads, Sequence Length[토큰갯수], d_k)
        WQ_splits = self.split_heads(WQ)
        WK_splits = self.split_heads(WK)
        WV_splits = self.split_heads(WV)

        # out (batch, heads, 50, 64), attention_weights (batch, heads, 50, 50)
        out, attention_weights = self.scaled_dot_product_attention(
            WQ_splits, WK_splits, WV_splits, mask)

        # (16, 50, 8, 64) -> (16, 50, 512)
        out = self.combine_heads(out)
        out = self.linear(out)

        return out, attention_weights

In [ ]:
class PoswiseFeedForwardNet(nn.Module):
    def __init__(self, d_model, d_ff):
        #d_ff:  2048, d_model: 512
        super(PoswiseFeedForwardNet, self).__init__()
        self.w_1 = nn.Linear(d_model, d_ff) # 512 -> 2048
        self.w_2 = nn.Linear(d_ff, d_model) # 예: 2048 -> 512
        self.relu = nn.ReLU() # "0보다 작은 숫자는 전부 0으로 만들어서 지워버리고, 양수(유의미한 정보)만 살려둬!"

    def forward(self, x):
        # x (batch_size, seq_len, d_model) (16, 50, 512)
        out = self.relu(self.w_1(x))
        out = self.w_2(out)

        return out


In [ ]:
class EncoderLayer(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, dropout):
        super(EncoderLayer, self).__init__()

        self.enc_self_attn = MultiHeadAttention(d_model, n_heads)
        self.ffn = PoswiseFeedForwardNet(d_model, d_ff)

        # 레이어 정규화
        # 나누는 값(분산) 밑바닥에 항상 0.000001을 더해준다. 그러면 분산이 우연히 0이 되더라도, 
        # $0 + 0.000001 = 0.000001$로 나누게 되므로 프로그램이 죽지 않고 무사히 살아남게 됩
        self.norm_1 = nn.LayerNorm(d_model, eps=1e-6)
        self.norm_2 = nn.LayerNorm(d_model, eps=1e-6)

        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask):
        """
        Multi-Head Attention
        """
        # x (16, 50, 512) : positional encodingehls embedding vector
        # mask: Padding Mask (16, 1, 1, 50) 
        # 가짜 단어([PAD])가 있는 자리는 1, 진짜 단어가 있는 자리는 0
        # Pre-LayerNorm: 논문과는 다름...덕분에 웜업의 '필수성'은 크게 사라졌다
        residual = x
        #  512짜리 빵틀에 (16, 50, 512) 모양의 반죽 덩어리를 밀어넣는다...내부에서 for문으로 돌린다..
        out = self.norm_1(x)

        # out: context vector (16, 50, 512)
        # enc_attn: 멀티헤드 어텐션 메트릭스...(batch, heads, 50, 50)
        out, enc_attn = self.enc_self_attn(out, out, out, mask)
        out = self.dropout(out)
        out += residual #잔차 연결

        """
        Position-Wise Feed Forward Network
        """
        residual = out # (16, 50, 512)
        out = self.norm_2(out)
        out = self.ffn(out) # (16, 50, 512)
        out = self.dropout(out)
        out += residual

        return out, enc_attn


슝=3


In [ ]:
class DecoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout):
        super(DecoderLayer, self).__init__()

        self.dec_self_attn = MultiHeadAttention(d_model, num_heads)
        self.enc_dec_attn = MultiHeadAttention(d_model, num_heads)

        self.ffn = PoswiseFeedForwardNet(d_model, d_ff)

        self.norm_1 = nn.LayerNorm(d_model, eps=1e-6)
        self.norm_2 = nn.LayerNorm(d_model, eps=1e-6)
        self.norm_3 = nn.LayerNorm(d_model, eps=1e-6)

        self.dropout = nn.Dropout(dropout)

    def forward(self, x, enc_out, causality_mask, padding_mask):
        """
        Masked Multi-Head Attention
        """
        residual = x
        out = self.norm_1(x)
        # out, dec_attn = self.dec_self_attn(out, out, out, padding_mask)
        out, dec_attn = self.dec_self_attn(out, out, out, causality_mask) 
        out = self.dropout(out)
        out += residual

        """
        Multi-Head Attention
        """
        residual = out
        out = self.norm_2(out)
        # out, dec_enc_attn = self.enc_dec_attn(out, enc_out, enc_out, causality_mask)
        out, dec_enc_attn = self.enc_dec_attn(out, enc_out, enc_out, padding_mask) 
        out = self.dropout(out)
        out += residual

        """
        Position-Wise Feed Forward Network
        """
        residual = out
        out = self.norm_3(out)
        out = self.ffn(out)
        out = self.dropout(out)
        out += residual

        return out, dec_attn, dec_enc_attn


In [12]:
class Encoder(nn.Module):
    def __init__(self, n_layers, d_model, n_heads, d_ff, dropout):
        super(Encoder, self).__init__()
        self.n_layers = n_layers
        self.enc_layers = nn.ModuleList([
            EncoderLayer(d_model, n_heads, d_ff, dropout)
            for _ in range(n_layers)
        ])

    def forward(self, x, mask):
        out = x
        enc_attns = []

        for i in range(self.n_layers):
            out, enc_attn = self.enc_layers[i](out, mask)
            enc_attns.append(enc_attn)

        return out, enc_attns


In [14]:
class Decoder(nn.Module):
    def __init__(self, n_layers, d_model, n_heads, d_ff, dropout):
        super(Decoder, self).__init__()
        self.n_layers = n_layers
        self.dec_layers = nn.ModuleList([
            DecoderLayer(d_model, n_heads, d_ff, dropout)
            for _ in range(n_layers)
        ])

    def forward(self, x, enc_out, causality_mask, padding_mask):
        out = x

        dec_attns = []
        dec_enc_attns = []
        for i in range(self.n_layers):
            out, dec_attn, dec_enc_attn = \
                self.dec_layers[i](out, enc_out, causality_mask, padding_mask)

            dec_attns.append(dec_attn)
            dec_enc_attns.append(dec_enc_attn)

        return out, dec_attns, dec_enc_attns

print("슝=3")

슝=3


In [ ]:
class Transformer(nn.Module):
    def __init__(self,
                 n_layers,
                 d_model,
                 n_heads,
                 d_ff,
                 src_vocab_size,
                 tgt_vocab_size,
                 pos_len,
                 dropout=0.2,
                 shared=True):
        super(Transformer, self).__init__()
        self.d_model = float(d_model)

        self.enc_emb = nn.Embedding(src_vocab_size, d_model)
        self.dec_emb = nn.Embedding(tgt_vocab_size, d_model)

        self.pos_encoding = self.positional_encoding(pos_len, d_model)
        # 1. GPU 이슈 해결: positional_encoding을 생성한 뒤 모델의 buffer로 등록해 줘야 합니다.
        pos_encoding = self.positional_encoding(pos_len, d_model)
        self.dropout = nn.Dropout(dropout)

        self.encoder = Encoder(n_layers, d_model, n_heads, d_ff, dropout)
        self.decoder = Decoder(n_layers, d_model, n_heads, d_ff, dropout)

        self.fc = nn.Linear(d_model, tgt_vocab_size)

        self.shared = shared
        if shared:
            # Word embedding 웨이트 공유 (tgt_vocab_size와 output 레이어가 맞아야 함)
            self.fc.weight = self.dec_emb.weight

    def positional_encoding(self, pos_len, d_model):
        position = torch.arange(0, pos_len).unsqueeze(1).float()
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * -(torch.log(torch.tensor(10000.0)) / d_model))
        pos_encoding = torch.zeros(pos_len, d_model)
        pos_encoding[:, 0::2] = torch.sin(position * div_term)
        pos_encoding[:, 1::2] = torch.cos(position * div_term)
        return pos_encoding

    def embedding(self, emb, x):
        seq_len = x.size(1)

         # 4. 방어 로직 (Assert): 입력된 시퀀스 길이가 예측된 시퀀스 길이 내에 있는지 확인
        assert seq_len <= self.pos_encoding.size(0), f"Sequence length ({seq_len}) exceeds configured positional encoding length ({self.pos_encoding.size(0)})."
        out = emb(x)

# 2 & 3. CPU 텐서 GPU 충돌 및 성능 문제 해결 & 논문 구현 기준
        # if self.shared:
        #     out *= torch.sqrt(torch.tensor(self.d_model))

        # 개선: shared 여부와 상관없이 math.sqrt 적용
        out *= math.sqrt(self.d_model)

        out += self.pos_encoding[:seq_len, :].unsqueeze(0)
        out = self.dropout(out)

        return out

    def forward(self, enc_in, dec_in, enc_mask, causality_mask, dec_mask):
        enc_in = self.embedding(self.enc_emb, enc_in)
        dec_in = self.embedding(self.dec_emb, dec_in)

        enc_out, enc_attns = self.encoder(enc_in, enc_mask)

        dec_out, dec_attns, dec_enc_attns = \
            self.decoder(dec_in, enc_out, causality_mask, dec_mask)

        logits = self.fc(dec_out)

        return logits, enc_attns, dec_attns, dec_enc_attns
